# CLARIMOL Training — NRP JupyterHub
RTX 2080 Ti (11GB) | 4-bit LoRA | Packing | fp16

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install dependencies
!pip install --user -r training_nrp/requirements.txt
!pip install --user -e .

In [ ]:
# 3. Verify imports
import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Configure training
from clarimol.train.config import TrainConfig

config = TrainConfig(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    model_max_length=512,
    data_dir="data/clarimol",
    output_dir="output/parsing_pretrain",
    batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=5e-4,
    epochs=1,
    lora_r=64,
    lora_alpha=16,
    packing=True,
    use_unsloth=False,  # avoid SDPA issues on consumer GPUs
    use_wandb=False,
    select_sample=10000,  # 10K/task = 50K total
    seed=42,
)
print(f"Config: batch={config.batch_size}, max_len={config.model_max_length}, packing={config.packing}")

In [ ]:
# Training
from clarimol.train.trainer import run_training
import logging
logging.basicConfig(level=logging.INFO)

model_path = run_training(config)
print(f"\nModel saved to: {model_path}")

In [ ]:
# Evaluation
from clarimol.eval.inference import evaluate_model

results = evaluate_model(
    model_path="output/parsing_pretrain/final",
    data_dir="data/test",
    use_unsloth=False,
    max_samples=500,
    batch_size=4,
)
for task, r in results.items():
    print(f"  {task:25s} acc={r.accuracy:.4f} ({r.correct}/{r.total})")